# <u>Gradient Boosting Classifier</u>

### Prerequisites:

* <a href="../6.Descision Trees Classifier/Descision Trees Classifier.ipynb">Check out the notebookes on Decision Tree Classifier</a>
* <a href="../7.Random Forest Classifier/Random Forest Classification.ipynb">Check out the notebooks on Random Forest Classification</a>

## Topics

* [1. What is Boosting?](#what)
* [2. Boosting vs. Bagging](#vs)
* [3. Forward Stagewise Additive Modeling](#forward)
* [4. Gradient Boosting](#gradient)
* [5. Regularization in Gradient Boosting](#reg)
* [6. Stochastic Gradient Boosting](#stochastic)
* [7. Gradient Boosting for Classification](#class)
* [8. Multiclass Gradient Boosting](#multi)
* [9. Gradient Boosting with Trees](#trees)
* [10. XGBoost (Extreme Gradient Boosting)](#xg)
* [11. XGBoost Regularization](#xg_reg)
* [12. Important XGBoost Hyperparameters](#xg_hyper)
* [13. Key Takeaways](#takeaway)

<a href="../../1.Regression/10.Gradient Boosting Regression/Gradient Boosting Regression.ipynb">Check out the notebook on Gradient Boosting Regression for more code</a>

In [1]:
import numpy as np # for arrays and random numbers
import matplotlib.pyplot as plt # for plotting
import plotly.express as px # for plotting
import plotly.graph_objects as go # for plotting

from sklearn.tree import (
    DecisionTreeClassifier, # for Classification Trees
    DecisionTreeRegressor # for Regression Trees
)

from sklearn.ensemble import (
    AdaBoostClassifier, # for AdaBoost in Classification
    AdaBoostRegressor, # for Adaboost in Regression
    GradientBoostingClassifier, # Gradient Boosting Classification
    GradientBoostingRegressor, # Gradient Boosting Regression
    HistGradientBoostingClassifier, # Histogram-based Gradient Boosting for Classification
    HistGradientBoostingRegressor, # Histogram-based Gradient Boosting for Regression
    RandomForestClassifier, # for Random Forest Classification
    RandomForestRegressor # for Random Forest Regression
)

# XGBoost
from xgboost import XGBClassifier, XGBRegressor

# LightGBM
#from lightgbm import LGBMClassifier, LGBMRegressor

# CatBoost
#from catboost import CatBoostClassifier, CatBoostRegressor

from sklearn.datasets import make_classification # toy data for classification
from sklearn.datasets import make_regression # toy data for regression

from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score

print("Setup complete")

Setup complete


<a class="anchor" id="what"></a>
# 1. What is Boosting?

- Boosting is an ensemble learning technique that combines many weak learners into a strong predictive model
- **Idea:** <u>Sequentially</u> apply a weak learner to <u>modified versions of the training data</u> with more focus on difficult observations
- Typical weak learners are decision trees and tree stumps (1-level decision trees)
- Boosting can be used fpr Classification, Regression and multiclass porblemns
- Weak learners are a prediction rule with a correct classification rate only slightly better than random guessing (so above 50% accuracy)

<a class="anchor" id="vs"></a>
# 2. Boosting vs. Bagging


<div style="display:flex; gap:20px;">

<div style="
padding:16px;
border-radius:8px;
width:50%;
">

### Bagging / Random Forests
- Base learners are decision trees
- Base learners are trained independently
- Equal weighting for base learners
- Goal: Mainly reduces variance
- Usually resistant to overfitting



</div>

<div style="
padding:16px;
border-radius:8px;
width:50%;
">

### Boosting
- Base learners are (most commonly) weak decision trees 
- Base learners are trained sequentially
- Following base learners focus on errors of previous base learners
- Different weighting for different base learners
- Goal: Reduces both bias and variance
- More prone to overfitting

</div>
</div>

<p align="center">
<img src="pics/1.png" width="500"/>
</p>


<a class="anchor" id="forward"></a>
# 3. Forward Stagewise Additive Modeling

**Gradient boosting builds a model additively:**

$$
f(x) = \sum_{m=1}^M \alpha^{[m]} b(x,\theta^{[m]})
$$

- $b(x,\theta^{[m]})$ is the $m$-th base learner
- $\alpha^{[m]}$ is weight of the $m$-th learner

Goal: Minimize empirical risk $\mathcal{R}_\text{emp}(f)=\sum_{i=1}^n L(y^{(i)},f(x^{(i)}))=\sum_{i=1}^n L(y^{(i)},\sum_{m=1}^M \alpha^{[m]} b(x^{(i)},\theta^{[m]}))$

**Instead of optimizing all learners at once, boosting:**
1. Starts with an initial model $\hat{f}^{[0]}(x)$
2. Adds one learner $b(x^{(i)},\theta)$ at a time and scales it with $\alpha \in [0,1]$ 
3. Greedily minimizes loss  at each step with respect to the nwxt additive component

$$
\min_{\alpha, \theta}\sum_{i=1}^n L(y^{(i)},\underbrace{\hat{f}^{[m-1]}(x^{(i)}) + \alpha b(x^{(i)},\theta)}_{\hat{f}^{[m]}})
$$

This process is called **forward stagewise additive modeling**.

Advantages:
- Easier optimization
- Natural regularization
- Enables early stopping


<a class="anchor" id="pseudo"></a>
# 4. Pseudo Residuals

$$
\tilde{r}^{(i)}=-\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}
$$

For squarred error loss (L2-loss):

$$
\tilde{r}^{(i)}=-\frac{\partial 0.5(y-f(x))}{\partial f(x)}=y-f(x)
$$

### Boosting as Gradient Descent

**We are at the model $f^{[m-1]}$ during minimization and at this point calculate the direction of the negative gradient also called the pseudo-residuals:**

$$
\hat{r}^{[m](i)}=-\left[\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}\right]_{f=f^{[m-1]}}
$$

The gradient descent update is:

$$
f^{[m]}(x^{(i)})=f^{[m-1]}(x^{(i)})-\alpha \frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f^{[m-1]}(x^{(i)})}
$$

- $\alpha \frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f^{[m-1]}(x^{(i)})}$ is movement of function $f$ in direction of data to reduce empirical risk




<a class="anchor" id="gradient_reg"></a>
# 4. Gradient Boosting for Regression

<p align="center">
<img src="pics/2.png" width="500"/>
</p>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/3.png" width="550"/>
  <img src="pics/4.png" width="550"/>
</div>


Gradient Boosting interprets boosting as gradient descent in function space.

## Main Idea

At each iteration:
1. Compute pseudo-residuals $\hat{r}^{[m](i)}$ (negative gradients of loss)
2. Fit a weak learner $b(x,\theta^{[m]})$ to these pseudo-residuals
$$
\hat{\theta}^{[m]}=\argmin_\theta \sum_{i=1}^n(\hat{r}^{[m](i)}-b(x^{(i)},\theta))^2
$$
3. Add learner to ensemble

For squared error loss:

$$
\hat{r}^{[m](i)}=y^{(i)}-\hat{f}^{[m-1]}(x^{(i)})
$$

These pseudo-residuals are ordinary residuals.

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/5.png" width="550"/>
  <img src="pics/6.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/7.png" width="550"/>
</div>

### Gradient Boosting Algorithm

1. Initialize $\hat{f}^{[0]}(x)=\underset{\theta_0 \in \mathbb{R}}{\argmin}\sum_{i=1}^n L(y^{(i)},\theta_0)$
2. for $m=1 \rightarrow M$ do
    - For all $i$: $\tilde{r}^{[m](i)}=\underbrace{-\left[\frac{\partial L(y,f)}{\partial f}\right]_{f=\hat{f}^{[m-1]}(x^{(i)}),y=y^{(i)}}}_{\text{Gradient}}$
    - Fit a regression base learner to the vector of pseudo-residuals $\tilde{r}^{[m]}$: $\hat{\theta}^{m}=\underset{\theta}{\argmin}\sum_{i=1}^n (\tilde{r}^{[m](i)}-b(x^{(i)},\theta))^2$
    - Set $\alpha^{[m]}$ to $\alpha$ or via line search
    - Update $\hat{f}^{[m]}=\hat{f}^{[m-1]}(x^{(i)}) + \alpha^{[m]} b(x,\hat{\theta}^{m})$
3. end for
4. Output: $\hat{f}(x)=\hat{f}^{[M]}(x)$


##### Line Search
- Learning rate $\alpha^{[m]}$ controls update size
- Commonly a learning rate of 0.1 is used for all base learners
- However we can also replace it by a line search

**Line Search is an iterative appoach to find a local minimum. So solve**

$$
\hat{\alpha}^{[m]}=\underset{\alpha}{\argmin} \sum_{i=1}^n L(y^{(i)},\hat{f}^{[m-1]}(x^{(i)}) + \alpha b(x,\hat{\theta}^{m}))
$$

### Gradient Boosting for Regression (Simple Example)

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Height (m)} & \textbf{Favorite Color} & \textbf{Gender} & \textbf{Weight (kg)} \\
\hline
1.6 & \text{Blue}  & \text{Male}   & 88 \\
\hline
1.6 & \text{Green} & \text{Female} & 76 \\
\hline
1.5 & \text{Blue}  & \text{Female} & 56 \\
\hline
1.8 & \text{Red}   & \text{Male}   & 73 \\
\hline
1.5 & \text{Green} & \text{Male}   & 77 \\
\hline
1.4 & \text{Blue}  & \text{Female} & 57 \\
\hline
\end{array}
$$

##### Goal: Predict continuous value "Weight".

##### Process: Fit a model to this training data with Gradient Boosting for Regression

##### Base Learner: Decision Trees for Regression

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/8.jpg" width="550"/>
  <img src="pics/9.jpg" width="550"/>
</div>

---
<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/10.jpg" width="550"/>
  <img src="pics/11.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/12.jpg" width="550"/>
  <img src="pics/13.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/15.jpg" width="550"/>
  <img src="pics/16.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/17.jpg" width="550"/>
  <img src="pics/18.jpg" width="550"/>
</div>


### Gradient Boosting for Regression (Detailed Example)

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Height (m)} & \textbf{Favorite Color} & \textbf{Gender} & \textbf{Weight (kg)} \\
\hline
1.6 & \text{Blue}  & \text{Male}   & 88 \\
\hline
1.6 & \text{Green} & \text{Female} & 76 \\
\hline
1.5 & \text{Blue}  & \text{Female} & 56 \\
\hline
\end{array}
$$


##### Goal: Predict continuous value Weight.

##### Process: Fit a model to this training data with Gradient Boosting for Regression

##### Base Learner: Decision Trees for Regression

**Note: Since we only have 3 samples the resulting trees in Gradient Boosting with Regression Trees will only have two leaves.**

### Gradient Boosting Algorithm with Regression Trees as base learners

Input: Data $\left\{(x^{(i)},y^{(i)})\right\}_{i=1}^n$ and a differentiable Loss function $L(y,f(x))$
1. Initialize model with a constant value: $f_0(x)=\underset{\theta}{\argmin}\sum_{i=1}^n L(y^{(i)},\theta)$
2. for $m=1$ to $M$ do
    - (A) Compute pseudo-residuals $r_{im}=\underbrace{-\left[\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}\right]_{f(x^{(i)})=f_{m-1}(x^{(i)})}}_{\text{Gradient}}$ for $i=1,\ldots,n$
    - (B) Fit a regression tree to the $r^{im}$ pseudo-residuals and create terminal regions $R_{jm}$ for $j=1,\ldots,J_m$ 

    - (C) For $j=1,\ldots,J_m$ compute $\theta_{jm}=\underset{\theta}{\argmin}\underset{x^{(i)} \in R_{jm}}{\sum} L(y^{(i)},f_{m-1}(x^{(i)})+\theta)$
    - (D) Update $f_m=f_{m-1}(x) + \alpha \sum_{j=1}^{J_m} \theta_{jm} \mathbb{I}(x \in R_{jm}) $
3. end for
4. Output: $f_{M}(x)$



<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/19.jpg" width="600"/>
</div>

---


<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/20.jpg" width="600"/>
</div>

---


<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/21.jpg" width="600"/>
</div>

---


<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/22.jpg" width="600"/>
  <img src="pics/23.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/24.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/25.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/26.jpg" width="600"/>
  <img src="pics/27.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/28.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/29.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/30.jpg" width="600"/>
</div>

### Gradient Boosting for Binary Classification (Simple Example)

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Likes Popcorn} & \textbf{Age} & \textbf{Favorite Color} & \textbf{Loves movies} \\
\hline
\text{Yes} & 12  & \text{Blue}   & \text{Yes} \\
\hline
\text{Yes} & 87 & \text{Green} & \text{Yes} \\
\hline
\text{No} & 44  & \text{Blue} & \text{No} \\
\hline
\text{Yes} & 19  & \text{Red}   & \text{No} \\
\hline
\text{No} & 32 & \text{Green}   & \text{Yes} \\
\hline
\text{No} & 14  & \text{Blue} & \text{Yes} \\
\hline
\end{array}
$$

##### Goal: Predict categorical/binary target "Loves Movies" 

##### Process: Fit a model to this training data with Gradient Boosting for Classification

##### Base Learner: Decision Trees for Regression

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/31.jpg" width="550"/>
  <img src="pics/32.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/33.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/34.jpg" width="550"/>
  <img src="pics/35.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/36.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/37.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/38.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/39.jpg" width="600"/>
</div>



### Gradient Boosting Algorithm with Binary Classification (Detailed Example)

Input: Data $\left\{(x^{(i)},y^{(i)})\right\}_{i=1}^n$ and a differentiable Loss function $L(y,f(x))$
1. Initialize model with a constant value: $f_0(x)=\underset{\theta}{\argmin}\sum_{i=1}^n L(y^{(i)},\theta)$
2. for $m=1$ to $M$ do
    - (A) Compute pseudo-residuals $r_{im}=\underbrace{-\left[\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}\right]_{f(x^{(i)})=f_{m-1}(x^{(i)})}}_{\text{Gradient}}$ for $i=1,\ldots,n$
    - (B) Fit a regression tree to the $r^{im}$ pseudo-residuals and create terminal regions $R_{jm}$ for $j=1,\ldots,J_m$ 

    - (C) For $j=1,\ldots,J_m$ compute $\theta_{jm}=\underset{\theta}{\argmin}\underset{x^{(i)} \in R_{jm}}{\sum} L(y^{(i)},f_{m-1}(x^{(i)})+\theta)$
    - (D) Update $f_m=f_{m-1}(x) + \alpha \sum_{j=1}^{J_m} \theta_{jm} \mathbb{I}(x \in R_{jm}) $
3. end for
4. Output: $f_{M}(x)$



<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/4_.jpg" width="550"/>
  <img src="pics2/5.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/6.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/7.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/8.jpg" width="550"/>
  <img src="pics2/9.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/10.jpg" width="550"/>
  <img src="pics2/11.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/12.jpg" width="600"/>
</div>

---
<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/13.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/14.jpg" width="550"/>
  <img src="pics2/15.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/16.jpg" width="600"/>
</div>

---
<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/17.jpg" width="600"/>
</div>

---

In [ ]:
# Gradient Boost Binary Classification with CLassification Trees

<a class="anchor" id="reg"></a>
# 5. Regularization in Gradient Boosting

Gradient Boosting can easily overfit, so regularization is important.

Main regularization methods:

1. Number of Iterations $M$
- Fewer base learners (additive components) reduce overfitting (Early stopping)

2. Tree Depth
- When using decision trees as base learners limit their maximum depth 
- Limiting tree depth also controls interaction complexity
- Small depth $\rightarrow$ simpler models

3. Learning Rate (Shrinkage)
- Small learning rate $\alpha$ slows learning
- Usually improves generalization
- Requires more base learners though

**Typical tradeoff: Small learning rate & choose $M$ via cross validation.**

<a class="anchor" id="stochastic"></a>
# 6. Stochastic Gradient Boosting

Adds randomness to gradient boosting.

At each iteration:

- Train on a random subset of data

Benefits:

- Reduces overfitting
- Improves generalization
- Similar idea to bagging

Additional hyperparameter: Subsample ratio (size of random data sets)

<a class="anchor" id="class"></a>
# 7. Gradient Boosting for Classification

<a class="anchor" id="class"></a>
## 7.1 Gradient Boosting for Binary Classification

For binary classification, Gradient Boosting uses classification loss functions.

Most common:
- Bernoulli / Logistic loss

Pseudo-residuals become:

$$
\hat{r}^{[m](i)}= y^{(i)}-\pi(x^{(i)})
$$

where:
- $\pi(x^{(i)}) = \frac{\exp(\log(\text{odds}))}{1+\exp(\log(\text{odds}))}$ = predicted probability

Important notes:
- Classification trees are fitted to pseudo-residuals
- Predicted probabilities come from logistic transformation



<a class="anchor" id="class"></a>
## 7.2 Gradient Boosting for Multiclass Classification

For $g$ classes meaning $y \in \{1,\ldots,g\}$:

- Build one discriminant function per class

Probabilities computed using softmax:

$$
\pi_k(x)=\frac{\exp(f_k(x))}{\sum_{j=1}^g \exp(f_j(x))} \text{ for } k \in \{1,\ldots,g\}
$$

Use Multiomial loss:

$$
L(y,f_1(x),\ldots,f_g(x))=-\sum_{k=1}^g \mathbb{I}_{\{y^{(i)}=k\}} \ln(\pi_k(x))
$$

Pseudo-residuals:

$$
-\frac{\partial L(y,f_1(x),\ldots,f_g(x))}{\partial f_k(x)} = \mathbb{I}_{\{y^{(i)}=k\}} - \pi_k(x)
$$

Process:
- Compute class probabilities
- Compute pseudo-residuals
- Fit trees for each class on pseudo-residuals
- Update all class models

### Gradient Boosting for Multiclass CLassification

1. Initialize $f_k^{[0]}(x)=0, k = 1,\ldots,g$

2. for $m=1$ to $M$ do
    - Set $\pi_k^{[m]}(x)=\frac{\exp(f_k^{[m]}(x))}{\sum_{j=1}^g\exp(f_j^{[m]}(x))}$, $k=1,\ldots,g$
    - for $k=1$ to $g$ do
        - For all $i$: Compute $\tilde{r}_k^{[m](i)}=\mathbb{I}_{\{y^{(i)}=k\}} - \pi_k^{[m]}(x^{(i)})$
        
        - Fit a regression base learner $\hat{b}_k^{[m]}$ to the pseudo-residuals $\tilde{r}_k^{[m](i)}$

        - Update $\hat{f}_k^{[m]}=\hat{f}_k^{[m-1]}+\alpha\hat{b}_k^{[m]}$
    - end for
3. end for

4. Output $\hat{f}_1^{[M]},\ldots,\hat{f}_g^{[M]}$

<a class="anchor" id="trees"></a>
# 9. Gradient Boosting with Trees

Decision trees are the most common base learners.

Advantages:
- Handle categorical data
- Robust to outliers
- Handle missing values
- Fast training
- Automatic variable selection
- Interaction Depth

Depth of tree $b^{[m]}(x)$ determines interaction complexity.

$$
f(x)=\sum_{m=1}^M \alpha^{[m]}b^{[m]}(x)
$$

- Depth 1 (stumps):
    - additive model only
    - no feature interactions
    - $f(x)=f_0 + \sum_{j=1}^p f_j(x_j)$
    - $f_0$ being constant intercept
<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/18.png" width="300"/>
</div>

- Depth 2+:
    - captures feature interactions
    - $f(x)=f_0 + \sum_{j=1}^p f_j(x_j) + \sum_{j \neq k} f_{j,k}(x_j,x_k)$
    - $f_0$ being constant intercept
<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/19.png" width="300"/>
</div>

Gradient Boosting with trees is highly flexible and powerful.

- Trees as additive model $b(x)=\sum_{t=1}^T c_t \mathbb{I}_{\{x \in R_t\}}$
- $R_t$ are terminal regions and $c_t$ are terminal constants

$$
\begin{align*}
f^{[m]}(x)
&=f^{[m-1]}(x) + \alpha^{[m]}b^{[m]}(x) \\
&=f^{[m-1]}(x) + \alpha^{[m]} \sum_{t=1}^{T^{[m]}} c_t^{[m]} \mathbb{I}_{\{x \in R_t^{[m]}\}} \\
&=f^{[m-1]}(x) + \alpha^{[m]} \sum_{t=1}^{T^{[m]}} \tilde{c}_t^{[m]} \mathbb{I}_{\{x \in R_t^{[m]}\}} \\
\end{align*}
$$

- $\tilde{c}_t^{[m]}=\alpha^{[m]} \cdot c_t^{[m]}$ if $\alpha^{[m]}$ is a constant learning rate
- Fit the regression tree against the pseudo-residuals $\tilde{r}^{[m](i)}$ adapt terminal constants (in leafs) to become more loss optimal

$$
\tilde{c}_t^{[m]} = \argmin_c \sum_{x^{(i)} \in R_t^{[m]}} L(y^{(i)}, f^{[m-1]}(x^{(i)}) + c)
$$

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics2/20.png" width="450"/>
</div>

- There is no closed form solution for solving $\tilde{c}_t^{[m]} = \underset{c}{\argmin} \sum_{x^{(i)} \in R_t^{[m]}} L(y^{(i)}, f^{[m-1]}(x^{(i)}) + c)$ and finding $\hat{c}_{tk}^{[m]}$
- So approximate solution with single Newton-Raphson step
- Decomposes the problem into a separate calculation for each terminal node

$$
\hat{c}_{tk}^{[m]}=\frac{g-1}{g} \frac{\sum_{x^{(i)} \in R_{tk}^{[m]}} \tilde{r}_k^{[m](i)}}{\sum_{x^{(i)} \in R_{tk}^{[m]}}  \left | \tilde{r}_k^{[m](i)} \right | \left(1 - \left | \tilde{r}_k^{[m](i)} \right | \right)}
$$

### Gradient Boosting for Multiclass CLassification with Trees

1. Initialize $f_k^{[0]}(x)=0, k = 1,\ldots,g$

2. for $m=1$ to $M$ do
    - Set $\pi_k^{[m]}(x)=\frac{\exp(f_k^{[m]}(x))}{\sum_{j=1}^g\exp(f_j^{[m]}(x))}$, $k=1,\ldots,g$
    - for $k=1$ to $g$ do
        - For all $i$: Compute $\tilde{r}_k^{[m](i)}=\mathbb{I}_{\{y^{(i)}=k\}} - \pi_k(x^{(i)})$
        
        - Fit a regression base learner to the pseudo-residuals $\tilde{r}_k^{[m](i)}$ giving terminal regions $R_{tk}^{[m]}$

        - Compute
        $$
        \hat{c}_{tk}^{[m]}=\frac{g-1}{g} \frac{\sum_{x^{(i)} \in R_{tk}^{[m]}} \tilde{r}_k^{[m](i)}}{\sum_{x^{(i)} \in R_{tk}^{[m]}}  \left | \tilde{r}_k^{[m](i)} \right | \left(1 - \left | \tilde{r}_k^{[m](i)} \right | \right)}
        $$

        - Update $\hat{f}_k^{[m]}=\hat{f}_k^{[m-1]}+\sum_{t} \hat{c}_{tk}^{[m]}\mathbb{I}_{\{x \in R_{tk}^{[m]}\}}$

    - end for
3. end for

4. Output $\hat{f}_1^{[M]},\ldots,\hat{f}_g^{[M]}$

<a class="anchor" id="xg"></a>
# 10. XGBoost (Extreme Gradient Boosting)

XGBoost is an optimized implementation of Gradient Boosting.

Main features:
- Fast and scalable
- Parallelized tree construction
- GPU support
- Regularization
- Subsampling
- Efficient split finding

Widely used in:
- Structured/tabular data tasks
- Industry ML systems

<a class="anchor" id="xg_reg"></a>
# 11. XGBoost Regularization

XGBoost introduces additional regularization terms.

$$
\mathcal{R}_{\text{reg}}^{[m]}=\sum_{i=1}^n L(y^{(i)}, f^{[m-1]}(x^{(i)}) + b^{[m]}(x^{(i)})) + \lambda_1 J_1(b^{[m]}) + \lambda_2 J_2(b^{[m]}) + \lambda_3 J_3(b^{[m]})
$$

Penalties
1. Tree complexity penalty
    - $J_1(b^{[m]})=T^{[m]}$
    - penalizes tree depth by penalizing a number of leaves
    - $T^{[m]}$ is number of leaves to penalize

2. L2 regularization
    - $J_2(b^{[m]})=\lVert c^{[m]} \rVert_2^2$
    - shrinks leaf weights
    - $\lVert c^{[m]} \rVert_2^2$ is L2 penalty over leaf values


3. L1 regularization
    - $J_3(b^{[m]})=\lVert c^{[m]} \rVert_1$
    - $\lVert c^{[m]} \rVert_1$ is L1 penalty over leaf values (less constants per leaf)
    - encourages sparsity

Benefits:
- Better generalization
- Reduced overfitting
- More stable trees

XGBoost also uses:
- feature subsampling
- data subsampling
- pruning (grow full tree and then remove risk expensive splits)
- dropout-style boosting (DART)


<a class="anchor" id="xg_hyper"></a>
# 12. Important XGBoost Hyperparameters

```python
# Installation
# pip install xgboost



# 1. XGBoost (Classification & Regression)

from xgboost import XGBClassifier, XGBRegressor



# Classification

xgb_clf = XGBClassifier(
    n_estimators=100,       # number of boosting rounds
    learning_rate=0.1,      # shrinkage
    max_depth=3,            # tree depth
    subsample=0.8,          # row sampling
    colsample_bytree=0.8,   # feature sampling
    objective="binary:logistic", # binary classification
    eval_metric="logloss",
    random_state=42
)

xgb_clf.fit(X, y)

xgb_clf.predict(X)          # predicted labels
xgb_clf.predict_proba(X)    # class probabilities
xgb_clf.score(X, y)         # accuracy

xgb_clf.feature_importances_
xgb_clf.get_booster()



# Regression

xgb_reg = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_reg.fit(X, y)

xgb_reg.predict(X)          # predicted values
xgb_reg.score(X, y)         # R^2 score

xgb_reg.feature_importances_
xgb_reg.get_booster()



# 2. Train/Test Split Evaluation

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    mean_absolute_error
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Classification Evaluation

clf = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy_score(y_test, y_pred)



# Regression Evaluation

reg = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)

mean_squared_error(y_test, y_pred)
mean_absolute_error(y_test, y_pred)



# 3. Hyperparameter Tuning (Grid Search)

from sklearn.model_selection import GridSearchCV



# Classification Grid Search

param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.3],
    "max_depth": [2, 4, 6],
    "subsample": [0.8, 1.0]
}

grid_clf = GridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

grid_clf.fit(X, y)

grid_clf.best_params_
grid_clf.best_score_
grid_clf.best_estimator_


# Regression Grid Search

grid_reg = GridSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=42
    ),
    param_grid={
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 0.3],
        "max_depth": [2, 4, 6],
        "subsample": [0.8, 1.0]
    },
    scoring="r2",
    cv=5,
    n_jobs=-1
)

grid_reg.fit(X, y)

grid_reg.best_params_
grid_reg.best_score_
grid_reg.best_estimator_



# 4. Feature Importance

import pandas as pd

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_clf.feature_importances_
})

importance_df.sort_values(
    by="importance",
    ascending=False
)



# 5. Early Stopping

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

early_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=4,
    random_state=42,
    early_stopping_rounds=20
)

early_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

early_model.best_iteration
early_model.best_score



# 6. Cross-Validation with xgboost API

import xgboost as xgb

dtrain = xgb.DMatrix(X, label=y)

params = {
    "objective": "reg:squarederror",
    "max_depth": 4,
    "eta": 0.1
}

cv_results = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=100,
    nfold=5,
    metrics="rmse",
    early_stopping_rounds=10,
    seed=42
)

cv_results.head()



# 7. Plot Feature Importance

import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(xgb_clf)

plt.show()



# 8. Plot Trees

from xgboost import plot_tree

plt.figure(figsize=(20, 10))

plot_tree(
    xgb_clf,
    num_trees=0
)

plt.show()



# 9. Save and Load Models

# Save model
xgb_clf.save_model("xgb_classifier.json")

# Load model
new_model = XGBClassifier()

new_model.load_model("xgb_classifier.json")



# 10. Important Hyperparameters

# n_estimators
# -> number of boosting rounds

# learning_rate (eta)
# -> contribution of each tree

# max_depth
# -> tree complexity

# subsample
# -> fraction of rows sampled

# colsample_bytree
# -> fraction of features sampled

# gamma
# -> minimum split loss reduction

# min_child_weight
# -> minimum sum of instance weights in child

# reg_alpha
# -> L1 regularization

# reg_lambda
# -> L2 regularization



# 11. Common Objectives

# Classification
# "binary:logistic"
# "multi:softprob"
# "multi:softmax"

# Regression
# "reg:squarederror"
# "reg:absoluteerror"

# Ranking
# "rank:pairwise"



# 12. GPU Training
gpu_model = XGBClassifier(
    tree_method="hist",
    device="cuda",
    random_state=42
)

gpu_model.fit(X, y)


# 13. Important Notes

# - XGBoost = Extreme Gradient Boosting
# - Sequentially builds trees to correct previous errors
# - Uses gradient boosting framework
# - Includes L1/L2 regularization
# - Handles missing values automatically
# - Supports parallel computation and GPU training
# - Usually performs extremely well on tabular data
# - Smaller learning_rate usually requires more trees
# - Deep trees may overfit
# - Subsampling often improves generalization
# - Feature importance is built-in
# - Early stopping is highly recommended
```


<a class="anchor" id="takeaway"></a>
# 13. Key Takeaways

- Boosting sequentially combines weak learners into a strong learner
- AdaBoost increases focus on misclassified observations
- Gradient Boosting performs gradient descent in function space
- Pseudo-residuals guide learning
- Trees are the dominant base learner
- Regularization is crucial for avoiding overfitting
- XGBoost is a highly optimized and widely used boosting framework
- Learning rate, tree depth, and number of iterations are the most important tuning parameters